# Neural Half-Cheetah — Full Training (Colab)

Runs the full 5000-episode 2-phase training with stabilization improvements:
- Phase-aware epsilon reset at Phase 1→2 boundary
- Buffer truncation (remove stale diverse data)
- Cosine LR annealing with warm restart
- Gradient clipping on reward network

**Requires MuJoCo** (installed automatically below).

**Runtime**: ~2-3 hours on CPU, ~45-60 min with GPU

In [ ]:
# 1. Install MuJoCo and dependencies
!pip install -q mujoco gymnasium[mujoco] torch numpy scipy matplotlib scikit-learn imageio imageio-ffmpeg

# Clone repo
!git clone -b neural-successor-representation https://github.com/PrashRangarajan/Successor_Active_Inference_Clean.git
%cd Successor_Active_Inference_Clean

In [ ]:
# 2. Verify MuJoCo + imports
import sys
sys.path.insert(0, '.')

import torch
import gymnasium as gym
import numpy as np

# Test MuJoCo environment
env = gym.make('HalfCheetah-v4')
obs, _ = env.reset()
print(f'HalfCheetah obs shape: {obs.shape}')
print(f'Action space: {env.action_space}')
env.close()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
print(f'PyTorch: {torch.__version__}')

In [ ]:
# 3. Run full training (5000 episodes: 2000 diverse + 3000 mixed)
import torch
device_flag = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training on: {device_flag}')
!python -u examples/run_neural_half_cheetah.py --train --device {device_flag}

In [ ]:
# 4. Display training figures
import os
from IPython.display import Image, display
import glob

fig_dir = 'figures/eval/neural_half_cheetah'
for img_path in sorted(glob.glob(f'{fig_dir}/*.png')):
    print(f'\n{os.path.basename(img_path)}')
    display(Image(filename=img_path, width=600))

In [ ]:
# 5. Print eval metrics
import numpy as np
import os

eval_dir = 'data/eval/neural_half_cheetah'
for f in sorted(os.listdir(eval_dir)):
    if f.endswith('.npy'):
        data = np.load(os.path.join(eval_dir, f))
        print(f'{f}: shape={data.shape}, mean={np.mean(data):.3f}, std={np.std(data):.3f}')

In [ ]:
# 6. Download results
from google.colab import files
import shutil

shutil.make_archive('neural_cheetah_results', 'zip', '.', 'data/neural_half_cheetah')
shutil.make_archive('neural_cheetah_eval', 'zip', '.', 'data/eval/neural_half_cheetah')
shutil.make_archive('neural_cheetah_figures', 'zip', '.', 'figures/eval/neural_half_cheetah')

files.download('neural_cheetah_results.zip')
files.download('neural_cheetah_eval.zip')
files.download('neural_cheetah_figures.zip')